# Sesión 1 — Por qué una figura nos engaña o nos ayuda a decidir

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 1 de 9  
**Tipo:** Notebook guiado (caso del docente: Golden State Realty Advisors)

El curso empieza con un informe que salió mal: el trimestre pasado, el comité de inversión de Golden State tomó una
decisión guiado por un gráfico erróneo. Este notebook reproduce, paso a paso, cada cifra y cada gráfico de la
presentación de la sesión.

## Ubicación del curso

| Sesión | Pregunta del comité | Lo que se aprende |
|---|---|---|
| **S1** | **¿Por qué el informe del trimestre pasado nos llevó a una mala decisión?** | **Diagnosticar figuras; fijar quién decide qué** |
| S2 | ¿Qué muestran los datos sobre nuestra pregunta? | El gráfico según el tipo de dato |
| S3 | ¿Podemos confiar en lo que vemos? | Calidad y modelo de datos |
| S4 | ¿Qué está pasando, por qué importa y qué hacemos? | Hallazgo verificado, SCQA y gráfico principal |
| S5 | ¿Dónde ocurre y a qué escala? | Mapas y normalización |
| S6 | ¿Qué necesitamos ver cada trimestre? | El diseño del tablero |
| S7 | ¿Puede la IA ayudarnos sin engañarnos? | Del notebook a Power BI: el modelo, las medidas y la IA como asistente |
| S8 | ¿Es honesto y responsable? | El tablero final: accesible, honesto y listo para el comité |
| S9 | ¿Se aprueba la recomendación? | Sustentación ante el comité |

El curso es la preparación de una reunión de comité: de la S1 a la S8 el equipo prepara su propuesta y en la S9 la
sustenta. De la S1 a la S6 se trabaja en Python, que es la fuente de las cifras. Programación web de visualizaciones, otras herramientas de tableros y certificaciones no se cubren.

## Agenda de la sesión

| Parte | Contenido |
|---|---|
| Ubicación | El curso en nueve sesiones y cómo funciona |
| Parte 1 | El informe del trimestre pasado: figuras feas, malas y erróneas, y sus correcciones |
| Parte 2 | Datos y estéticas |
| Parte 3 | Escalas lineales y logarítmicas |
| Parte 4 | Coordenadas: un mapa no es un plano cartesiano |
| Parte 5 | Contexto: quién decide qué (la ficha del comité) |
| Cierre | Errores frecuentes, síntesis y presentación del taller |

## Objetivos de aprendizaje

1. Diagnosticar una figura como fea, mala o errónea y explicar por qué (Wilke, 2019, pp. 1–3).
2. Reconocer los gráficos que conviene evitar en un informe de negocio: torta, 3D y doble eje (Knaflic, 2015, pp. 61–68).
3. Identificar el tipo de cada variable y la estética que la representa (Wilke, 2019, pp. 7–12).
4. Elegir entre una escala lineal y una logarítmica, y reconocer que latitud y longitud no son un plano cartesiano (Wilke, 2019, pp. 13–24).
5. Fijar el destinatario, la decisión y la pregunta antes de graficar (Knaflic, 2015, pp. 20–26).

## Problemática de negocio

**Golden State Realty Advisors** asesora a un fondo de inversionistas en vivienda de California. Cada trimestre, su
comité de inversión decide en qué distritos hace el equipo su debida diligencia, con capacidad para **150 distritos**.

**Qué pasó el trimestre pasado.** El analista anterior entregó un informe de cuatro gráficos. Uno de ellos hacía ver
la vivienda cerca de la bahía mucho más cara que la del interior, y el comité concentró allí su atención. Este
trimestre el comité pide al nuevo equipo revisar ese informe antes de confiar en un análisis nuevo.

Licencia del caso: los datos son del censo de 1990 (Pace y Barry, 1997) y se tratan como actuales; se discute en la S8.

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
FUENTE_GS = "Fuente: California Housing, censo de 1990 (Pace y Barry, 1997). Cálculos del curso."
df = pd.read_csv(DATOS / "housing.csv")
ZONAS = {"INLAND": "Interior", "<1H OCEAN": "A menos de 1 h del mar", "NEAR OCEAN": "Cerca del mar",
         "NEAR BAY": "Cerca de la bahía", "ISLAND": "Isla"}
df["zona"] = df["ocean_proximity"].map(ZONAS)
print(f"{len(df):,} distritos y {df.shape[1]} columnas")
display(df.head())

## Parte 1 — El informe del trimestre pasado

Wilke (2019, pp. 1–3) clasifica las figuras defectuosas en tres grupos:

| Tipo | Qué falla | Ejemplo de Wilke (p. 3) |
|---|---|---|
| **Fea** | Lo estético: funciona, pero incomoda | Colores demasiado vivos, cuadrícula prominente, tres tipos de letra |
| **Mala** | La percepción: induce a una lectura equivocada | Cada barra con su propia escala: los valores parecen más cercanos de lo que son |
| **Errónea** | Los datos: lo que se ve no corresponde a los valores | Sin escala explícita, barras de 3, 5 y 4 parecen de 1, 3 y 2 |

«A data visualization first and foremost has to accurately convey the data. It must not mislead or distort»
(Wilke, 2019, p. 1). Knaflic (2015, pp. 61–68) agrega los gráficos que conviene evitar: torta, dona, 3D y doble eje.

In [ ]:
valor_zona = df[df["zona"] != "Isla"].groupby("zona")["median_house_value"].mean()
ingreso_zona = df[df["zona"] != "Isla"].groupby("zona")["median_income"].mean() * 10_000
orden = valor_zona.sort_values().index
EJE_RECORTADO = 100_000
real = valor_zona.max() / valor_zona.min()
aparente = (valor_zona.max() - EJE_RECORTADO) / (valor_zona.min() - EJE_RECORTADO)
print(f"Valor medio: la zona más cara vale {real:.1f} veces la más barata; con el eje en {EJE_RECORTADO:,} parece {aparente:.1f} veces")
display(valor_zona.loc[orden].round(0).to_frame("valor medio (USD)"))

In [ ]:
# Figura 1 del informe: fea (se dibuja así a propósito).
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(df["median_house_value"], bins=30, color="#FF00FF", edgecolor="#00FF00", linewidth=1.5)
    ax.set_facecolor("#FFFF99"); ax.grid(True, color="black", linewidth=1.2)
    ax.set_title("Distribución del valor", fontfamily="serif", fontsize=16, color="red")
    ax.set_xlabel("median_house_value", fontfamily="monospace", fontsize=11)
    ax.set_ylabel("Frecuencia", fontsize=13, fontstyle="italic")
    guardar(fig, "informe_fea.png")

In [ ]:
# Corrección del gráfico feo: los mismos datos y los mismos 30 intervalos, con la paleta del curso, sin cuadrícula
# pesada, una sola tipografía, ejes en USD y un título que dice lo que se ve.
valor = df["median_house_value"]
q1, q3 = valor.quantile([0.25, 0.75])
en_maximo = int((valor == valor.max()).sum())
fig, ax = plt.subplots(figsize=(7, 4))
n, bordes, barras = ax.hist(valor, bins=30, color=PALETA["neutro"], edgecolor="white", linewidth=0.8)
barras[-1].set_facecolor(PALETA["acento"])            # el último intervalo: el valor máximo del archivo
ax.annotate(f"Incluye {miles(en_maximo)} distritos\nen el valor máximo", xy=(bordes[-2], n[-1]), xytext=(bordes[-2] - 150_000, n[-1] + 150),
            fontsize=10.5, color=PALETA["acento"], arrowprops={"arrowstyle": "-", "color": PALETA["acento"]})
ax.axvspan(q1, q3, color=PALETA["principal"], alpha=0.12, linewidth=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: miles(v / 1000)))
ax.set_xlabel("Valor mediano de la vivienda por distrito (miles de USD)"); ax.set_ylabel("Distritos")
limpiar_ejes(ax, x=False)
titulo(fig, f"La mitad de los distritos vale entre USD {miles(q1, -3)} y {miles(q3, -3)}",
       f"Franja: el 50 % central. En el extremo, {miles(en_maximo)} distritos tienen exactamente el valor máximo")
fuente(fig, FUENTE_GS)
guardar(fig, "corregida_fea.png")

In [ ]:
# Figura 2 del informe: mala (doble eje, con escalas elegidas para que las dos series parezcan moverse juntas).
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(list(orden), valor_zona.loc[orden], color="#4C72B0")
    ax2 = ax.twinx()
    ax2.plot(list(orden), ingreso_zona.loc[orden], color="#DD8452", marker="o", linewidth=3)
    ax.set_ylim(0, valor_zona.max() * 1.1); ax2.set_ylim(ingreso_zona.min() * 0.97, ingreso_zona.max() * 1.01)
    ax.set_ylabel("Valor medio"); ax2.set_ylabel("Ingreso medio")
    ax.set_title("Valor e ingreso por zona"); ax.tick_params(axis="x", labelsize=9.5)
    guardar(fig, "informe_mala.png")

In [ ]:
# Figura 3 del informe: errónea (el eje empieza en 100 000 y no muestra la escala).
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(list(orden), valor_zona.loc[orden], color="#4C72B0")
    ax.set_ylim(EJE_RECORTADO, valor_zona.max() * 1.03); ax.set_yticks([])
    ax.set_title("Valor de la vivienda por zona"); ax.tick_params(axis="x", labelsize=9.5)
    guardar(fig, "informe_erronea.png")

**Por qué engaña la figura 3.** Una barra se lee por su longitud. Si el eje empieza en USD 100 000, a cada barra se le
quita la misma cantidad, pero esa cantidad pesa muy distinto: a la barra más baja le quita casi todo y a la más alta,
menos de la mitad. Como el informe no muestra la escala, el comité no tiene cómo darse cuenta.

In [ ]:
# La parte de cada barra que el eje recortado esconde (en gris) y la que el comité vio (en color).
fig, ax = plt.subplots(figsize=(9, 4.2))
x = list(range(len(orden)))
ax.bar(x, [EJE_RECORTADO] * len(orden), color=PALETA["neutro"], alpha=0.35, width=0.6)
ax.bar(x, valor_zona.loc[orden] - EJE_RECORTADO, bottom=EJE_RECORTADO, color=PALETA["principal"], width=0.6)
ax.axhline(EJE_RECORTADO, color=PALETA["acento"], linewidth=2, linestyle="--")
ax.text(-0.42, EJE_RECORTADO, "Aquí empezaba\nel eje del informe", color=PALETA["acento"], fontsize=11, ha="right", va="center")
for i, z in enumerate(orden):
    ax.text(i, EJE_RECORTADO / 2, f"{100 * EJE_RECORTADO / valor_zona[z]:.0f} %\nescondido", ha="center", va="center",
            fontsize=10.5, color=PALETA["secundario"])
    ax.text(i, valor_zona[z] + 4_000, f"USD {miles(valor_zona[z], -3)}", ha="center", fontsize=10.5)
ax.set_xticks(x, list(orden)); limpiar_ejes(ax); ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.set_ylim(0, valor_zona.max() * 1.12); ax.set_xlim(-1.5, len(orden) - 0.5)
titulo(fig, f"El eje recortado le esconde al interior el {100 * EJE_RECORTADO / valor_zona.min():.0f} % de su barra; "
       f"a la bahía, el {100 * EJE_RECORTADO / valor_zona.max():.0f} %",
       f"Por eso la bahía parece valer {aparente:.0f} veces el interior, cuando vale {decimal(real, 1)} veces")
fuente(fig, FUENTE_GS)
guardar(fig, "eje_recortado.png")

In [ ]:
# Figura 4 del informe: torta de distritos por zona (con sombra, como venía en el informe).
conteo = df["zona"].value_counts()
with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.pie(conteo, labels=conteo.index, shadow=True, startangle=90, textprops={"fontsize": 9.5})
    ax.set_title("Distritos por zona")
    guardar(fig, "informe_torta.png")

In [ ]:
# El informe completo, como lo vio el comité.
fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
for ax, nombre in zip(ejes.flat, ["informe_fea.png", "informe_mala.png", "informe_erronea.png", "informe_torta.png"]):
    ax.imshow(plt.imread(IMAGENES / nombre)); ax.set_axis_off()
titulo(fig, "El informe del trimestre pasado: cuatro gráficos, tres defectos distintos",
       "Así llegó al comité; la pregunta de hoy es qué falla en cada uno", y=1.03)
guardar(fig, "informe_completo.png")

**Diagnóstico.**

| Gráfico | Tipo | Por qué |
|---|---|---|
| Distribución del valor | **Feo** | Colores chillones, cuadrícula negra, tres tipos de letra. Los datos están bien, pero cuesta leerlo |
| Valor e ingreso por zona | **Malo** | Dos ejes con escalas distintas: la línea parece seguir a las barras porque así se eligieron las escalas (Knaflic, 2015, pp. 66–67) |
| Valor por zona | **Erróneo** | El eje empieza en 100 000 y no muestra la escala: la longitud de las barras no corresponde a los valores |
| Distritos por zona | **Discutible** | Knaflic (2015, pp. 61–65) desaconseja la torta; Wilke (2019, pp. 93–95) la defiende para fracciones simples. Con una tajada casi invisible (Isla, 5 distritos) y sombra, aquí no ayuda |

In [ ]:
# Corrección del gráfico erróneo: barras desde cero, con los valores y un título que dice la conclusión.
fig, ax = plt.subplots(figsize=(9, 3.6))
colores = [PALETA["principal"] if z == "Interior" else PALETA["neutro"] for z in orden]
ax.barh(list(orden), valor_zona.loc[orden], color=colores, height=0.6)
for z in orden:
    ax.text(valor_zona[z] + 3_000, z, f"USD {miles(valor_zona[z], -3)}", va="center", fontsize=11)
limpiar_ejes(ax); ax.set_xlim(0, valor_zona.max() * 1.25)
titulo(fig, f"La vivienda cerca de la bahía vale {decimal(real, 1)} veces la del interior, no {aparente:.0f} veces",
       "Valor medio de la vivienda por zona, con el eje desde cero")
fuente(fig, FUENTE_GS)
guardar(fig, "corregida_erronea.png")

In [ ]:
# Corrección del gráfico malo: una variable por panel, cada una con su escala desde cero.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, serie, nombre in ((a1, valor_zona, "Valor medio de la vivienda"), (a2, ingreso_zona, "Ingreso medio del hogar")):
    ax.barh(list(orden), serie.loc[orden], color=PALETA["principal"], height=0.6)
    for z in orden:
        ax.text(serie[z] * 1.02, z, f"USD {miles(serie[z], -3)}", va="center", fontsize=10.5)
    limpiar_ejes(ax); ax.set_xlim(0, serie.max() * 1.35); ax.set_title(nombre, loc="left", fontsize=12)
a2.set_yticks([])
titulo(fig, "El valor cambia mucho más entre zonas que el ingreso", "Una variable por panel en lugar de dos ejes", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "corregida_mala.png")

**Por qué esta figura lleva al indicador de la S2.** Si el valor de la vivienda dependiera solo de cuánto ganan sus
residentes, las dos barras cambiarían en la misma proporción entre zonas y una vivienda costaría los mismos años de
ingreso en todas partes. No es así: el valor cambia mucho más que el ingreso. Dividir uno entre otro dice cuántos años
de ingreso cuesta la vivienda en cada zona. Esa división es el indicador de la S2 (la razón precio/ingreso): donde es
baja, la vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene, que es la pregunta de la ficha.

In [ ]:
# Cuántos años de ingreso cuesta la vivienda en cada zona (valor medio / ingreso medio del hogar).
anios = (valor_zona / ingreso_zona).loc[orden]
print(anios.round(1).to_string())

In [ ]:
# Corrección de la torta: barras ordenadas con el porcentaje.
pct = 100 * conteo / conteo.sum()
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(list(pct.index[::-1]), pct.values[::-1], color=PALETA["principal"], height=0.6)
for z, v in pct.items():
    ax.text(v + 0.6, z, f"{decimal(v, 1)} % ({miles(conteo[z])})", va="center", fontsize=10.5)
limpiar_ejes(ax); ax.set_xlim(0, pct.max() * 1.35)
titulo(fig, "Casi la mitad de los distritos está a menos de una hora del mar",
       "Distritos por zona; Isla tiene 5 distritos y no se ve en una torta")
fuente(fig, FUENTE_GS)
guardar(fig, "corregida_torta.png")

### Punto de control 1 — El informe

**¿Cuál de los cuatro gráficos llevó al comité a una mala decisión y por qué?**
El erróneo: con el eje en 100 000 y sin escala, la bahía parecía valer unas seis veces el interior; vale el doble.

**¿Por qué el gráfico feo no es tan grave como el erróneo?**
Porque los datos están bien representados; incomoda, pero no engaña. Wilke advierte que la frontera entre feo y malo es difusa.

**¿Debemos prohibir las tortas?**
Los autores no coinciden: Knaflic las desaconseja y Wilke las defiende para fracciones simples. La pregunta es si esta torta ayuda a esta decisión.

> Busquen en su trabajo un gráfico con doble eje. ¿Las escalas se eligieron por los datos o por la historia que se quería contar?

## Parte 2 — Datos y estéticas

Toda visualización convierte valores de datos en **estéticas**: posición, forma, tamaño, color, grosor y tipo de
línea (Wilke, 2019, pp. 7–8). El tipo de variable decide qué estética y qué escala le sirven (Wilke, 2019, p. 9,
Tabla 2-1): **cuantitativa continua o discreta**, **cualitativa** (factor, con o sin orden), fecha u hora, y texto.

In [ ]:
tipos = pd.DataFrame([
    ("longitude, latitude", "Cuantitativa continua", "Posición (en un mapa)"),
    ("housing_median_age", "Cuantitativa discreta (años)", "Posición"),
    ("total_rooms, total_bedrooms, population, households", "Cuantitativa discreta (conteos)", "Posición o tamaño"),
    ("median_income", "Cuantitativa continua", "Posición o color"),
    ("median_house_value", "Cuantitativa continua", "Posición o color"),
    ("ocean_proximity", "Cualitativa (factor sin orden)", "Color o forma"),
], columns=["Columna", "Tipo de variable (Wilke, Tabla 2-1)", "Estética apropiada"])
display(tipos)

In [ ]:
# Figura: cuatro estéticas a la vez (posición x, posición y, color y tamaño).
muestra = df[df["zona"] != "Isla"].sample(1500, random_state=1)
colores_zona = {"Interior": PALETA["acento"], "A menos de 1 h del mar": PALETA["principal"],
                "Cerca del mar": PALETA["marino"], "Cerca de la bahía": PALETA["secundario"]}
fig, ax = plt.subplots(figsize=(9, 5))
for z, d in muestra.groupby("zona"):
    ax.scatter(d["median_income"] * 10, d["median_house_value"] / 1000, s=d["population"] / 60, alpha=0.5,
               color=colores_zona[z], label=z, linewidths=0)
ax.set_xlabel("Ingreso mediano del hogar (miles de USD)  ← posición x")
ax.set_ylabel("Valor mediano (miles de USD)  ← posición y")
ax.legend(title="Zona  ← color", loc="upper left", fontsize=9.5, markerscale=0.6)
titulo(fig, "Cuatro variables, cuatro estéticas: posición x, posición y, color y tamaño",
       "Muestra de 1 500 distritos; el tamaño del punto es la población")
fuente(fig, FUENTE_GS)
guardar(fig, "esteticas.png")

## Parte 3 — Escalas lineales y logarítmicas

Una **escala** asigna a cada valor de los datos un único valor de la estética (Wilke, 2019, p. 10). En un eje
logarítmico, distancias iguales representan **proporciones** iguales; sirve cuando los datos abarcan varios órdenes
de magnitud (Wilke, 2019, pp. 16–21).

In [ ]:
p = df["population"]
print(f"Población por distrito: mediana {p.median():,.0f}; percentil 99 {p.quantile(0.99):,.0f}; máximo {p.max():,.0f}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.hist(p, bins=60, color=PALETA["neutro"])
a1.set_title("Escala lineal: casi todo queda en la primera barra", loc="left", fontsize=11.5)
a1.set_xlabel("Habitantes por distrito")
a2.hist(p[p > 0], bins=np.logspace(0, np.log10(p.max()), 60), color=PALETA["principal"])
a2.set_xscale("log"); a2.set_title("Escala logarítmica: se ve la forma completa", loc="left", fontsize=11.5)
a2.set_xticks([1, 10, 100, 1_000, 10_000], ["1", "10", "100", "1 000", "10 000"])   # valores escritos, sin exponentes
a2.set_xlabel("Habitantes por distrito (escala logarítmica)")
for ax in (a1, a2):
    ax.set_yticks([]); ax.spines["left"].set_visible(False)
titulo(fig, "Cuando los datos abarcan varios órdenes de magnitud, conviene un eje logarítmico",
       "Distribución de la población por distrito", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "escala_log.png")

## Parte 4 — Coordenadas: un mapa no es un plano cartesiano

«Because the earth is a sphere, drawing latitude and longitude as Cartesian axes is misleading and not recommended»
(Wilke, 2019, p. 24). Un mapa usa una **proyección**. Para California, una proyección de áreas iguales (EPSG:3310).

In [ ]:
condados = gpd.read_file(DATOS / "cb_2018_us_county_20m.zip")
california = condados[condados["STATEFP"] == "06"]
puntos = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
# Círculos de 100 km de radio, trazados en metros (EPSG:3310) y llevados a grados para el panel izquierdo.
centros = gpd.GeoSeries(gpd.points_from_xy([-123.2, -121.2, -119.6, -117.4, -116.5], [41.0, 38.7, 36.6, 34.9, 33.2]),
                        crs=4326)
circulos = gpd.GeoSeries(centros.to_crs(3310).buffer(100_000), crs=3310)
g, m = california.to_crs(4326).total_bounds, california.to_crs(3310).total_bounds
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5.8))
california.to_crs(4326).plot(ax=a1, color=PALETA["fondo"], edgecolor="white")
a1.scatter(df["longitude"], df["latitude"], s=1, color=PALETA["principal"], alpha=0.3)
circulos.to_crs(4326).plot(ax=a1, facecolor="none", edgecolor=PALETA["acento"], linewidth=2)
a1.set_aspect("equal")   # un grado de longitud se dibuja igual que uno de latitud
a1.set_xlabel(""); a1.set_ylabel("")   # geopandas rotula los ejes en grados; con Calibri esos rótulos no se dibujan bien
a1.set_title("Latitud y longitud como ejes (en grados)", loc="left", fontsize=11.5)
california.to_crs(3310).plot(ax=a2, color=PALETA["fondo"], edgecolor="white")
pp = puntos.to_crs(3310)
a2.scatter(pp.geometry.x, pp.geometry.y, s=1, color=PALETA["principal"], alpha=0.3)
circulos.plot(ax=a2, facecolor="none", edgecolor=PALETA["acento"], linewidth=2)
a2.set_axis_off(); a2.set_title("Proyección de áreas iguales (EPSG:3310)", loc="left", fontsize=11.5)
fig.canvas.draw()   # con el aspecto igual, la posición final de cada panel se conoce después de dibujar
for ax, texto in ((a1, f"{decimal(g[2] - g[0], 1)}° de ancho por {decimal(g[3] - g[1], 1)}° de alto: el estado sale más ancho"),
                  (a2, f"{miles((m[2] - m[0]) / 1000)} km de ancho por {miles((m[3] - m[1]) / 1000)} km de alto")):
    caja = ax.get_position()
    fig.text((caja.x0 + caja.x1) / 2, 0.02, texto, ha="center", fontsize=10.5, color=PALETA["secundario"])
titulo(fig, "Con latitud y longitud como plano, los círculos de 100 km salen ovalados y el estado, más ancho",
       "Los mismos distritos y los mismos cinco círculos de 100 km de radio, en dos sistemas de coordenadas", y=1.06)
fuente(fig, FUENTE_GS + " Límites: U.S. Census Bureau (2019).")
guardar(fig, "coordenadas.png")

In [ ]:
# Por qué: un grado de latitud mide casi lo mismo en todas partes, pero uno de longitud se encoge lejos del ecuador.
# En la esfera (radio medio de 6 371 km), un grado de longitud mide 111,2 km por el coseno de la latitud.
for lugar, lat in (("San Diego", 32.7), ("Centro de California", 37.0), ("Frontera con Oregón", 42.0), ("Bogotá", 4.6)):
    print(f"{lugar:22s} latitud {decimal(lat, 1)}°: un grado de longitud mide {decimal(111.2 * np.cos(np.radians(lat)), 1)} km")

En el panel izquierdo cada grado de longitud se dibuja como si midiera lo mismo que uno de latitud, así que el estado
se estira de este a oeste, más cuanto más al norte. En Bogotá, cerca del ecuador, la diferencia es mínima: para el mapa
de CONECTA importa poco, pero el principio vale para cualquier mapa que compare áreas o distancias (S5).

### Punto de control 2 — Estéticas, escalas y coordenadas

**¿Qué estética conviene para `ocean_proximity` y por qué no el tamaño?**
Color o forma: es una variable cualitativa sin orden; el tamaño sugiere que una zona es «más» que otra.

**¿Cuándo un eje logarítmico confunde en vez de ayudar?**
Cuando la audiencia no está acostumbrada: hay que rotularlo y decirlo en el título.

**¿Qué pasa si se grafican latitud y longitud como un plano?**
Se distorsionan la forma y las distancias, más cuanto más lejos del ecuador.

> ¿Qué variables de su proyecto tienen varios órdenes de magnitud?

## Parte 5 — Contexto: quién decide qué

Antes de graficar hay que poder responder dos preguntas: **¿quién es la audiencia?** y **¿qué necesita saber o
hacer?** (Knaflic, 2015, pp. 20–26). El informe del trimestre pasado no tenía destinatario ni decisión: mostraba
datos.

| Elemento | Golden State |
|---|---|
| Destinatario | Comité de inversión (perfil financiero, no técnico) |
| Decisión | En qué distritos hace el equipo la debida diligencia del trimestre |
| Restricción | 150 distritos por trimestre: lo que el equipo de debida diligencia alcanza a evaluar (supuesto del caso) |
| Por qué importa la restricción | De más de 20 000 distritos se puede revisar menos del 1 %: el análisis tiene que ordenar y elegir, no solo describir |
| Pregunta para el análisis | ¿Dónde la vivienda se vende por debajo de lo que el ingreso de sus residentes sostiene? |
| Qué se lleva a la S2 | Un indicador que responda esa pregunta y el gráfico apropiado para cada tipo de dato |

**Qué significa «lo que el ingreso sostiene».** El valor de una vivienda (un patrimonio) y el ingreso de un año (un
flujo) no se comparan directamente: se divide uno por otro y se obtiene **cuántos años de ingreso cuesta la
vivienda**. Un distrito interesa cuando ese número es **bajo frente a distritos comparables**. Eso es lo que el
análisis mide y puede comprobar con los datos.

**Lo que el análisis no prueba: la premisa del fondo.** El fondo apuesta a que esa brecha tiende a cerrarse y a que
por eso ahí la vivienda puede valorizarse. Es una premisa, no un hallazgo. Una razón baja también puede deberse a que
la zona siempre es más barata, a errores en los datos, a lo que el dato no mide (estado de las viviendas, colegios,
seguridad), a que el ingreso es el de quienes viven hoy y no el de quienes comprarían, o a que los datos son de 1990.
El curso enfrenta cada una de estas explicaciones: comparables (S4 y S5), calidad de los datos (S3), el potencial
como techo y el escenario (S4) y la revisión ética (S8).

## Errores frecuentes

1. **Confundir feo con erróneo.** Un gráfico con colores por defecto incomoda; uno con el eje recortado engaña.
2. **Elegir las escalas del doble eje para que cuenten una historia.** La relación entre dos series depende de cómo se escalen; mejor un panel por variable.
3. **Usar el tamaño o un color de intensidad para una variable sin orden.** Sugiere que una categoría es «más» que otra.
4. **Graficar latitud y longitud como un plano.** El mapa se deforma.
5. **Graficar antes de saber quién decide qué.** El informe muestra datos, pero no ayuda a decidir.

## Síntesis

| Qué hicimos | Por qué importa |
|---|---|
| Diagnosticamos el informe con tres criterios: feo, malo y erróneo | El erróneo llevó al comité a una mala decisión |
| Corregimos los tres gráficos defectuosos | El mismo dato, bien representado, cambia la lectura |
| Clasificamos las variables y sus estéticas | El tipo de dato decide la estética y la escala |
| Vimos escalas logarítmicas y proyecciones | Algunos datos y los mapas exigen otro sistema |
| Fijamos destinatario, decisión y pregunta | Ningún gráfico se hace sin saber para quién es |

**Próxima sesión:** qué gráfico corresponde a cada pregunta, y el indicador que responde la del comité.

## Transición al taller práctico

Cada equipo recibe el informe de su gerencia del trimestre pasado (CONECTA S.A., proyecto A: Retención),
lee el diccionario de datos, diagnostica el informe, lo corrige y llena la ficha de su proyecto en
`sesion_01_practica_proyecto_A.ipynb`.

## Referencias

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.

Pace, R. K., y Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters, 33*(3), 291–297. https://doi.org/10.1016/S0167-7152(96)00140-X

U.S. Census Bureau. (2019). *2018 cartographic boundary file, current county and equivalent for United States, 1:20,000,000* [Shapefile].

Wilke, C. O. (2019). *Fundamentals of data visualization: A primer on making informative and compelling figures*. O'Reilly Media.